In [ ]:
import math
import os
from typing import Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

In [ ]:
import crunch

crunch_tools = crunch.load_notebook()
train_data, test_data = crunch_tools.load_data()

In [ ]:
def train(
    datasets: List[Tuple[int, List[float], List[float], Optional[int]]],
    model_directory_path: str,
):
    """
    Fit a model.
    """

    # The baseline has no parameters, so we just save an empty placeholder artefact.
    model = {
        "note": "EWMA z-score detector has no learned parameters"
    }

    for dataset_id, x_hist, x_online, tau in datasets:
        ...

    joblib.dump(model, os.path.join(model_directory_path, "model.joblib"))

In [ ]:
def infer(
    datasets: Iterable[Tuple[List[float], Iterable[float]]],
    model_directory_path: str,
):
    """
    Stream per-step break probabilities using an EWMA z-score.

    alpha -- EWMA decay in (0, 1]; higher = faster response but more noise.
    kappa -- tanh scale
    """

    ALPHA = 0.05    # effective window ~ 1/alpha = 20 points
    KAPPA = 3.0     # |z| = 3 -> score ~ 0.76

    # Load the model artefact.
    # We don't actually have any parameters, but this shows how you would load them if you did.
    model = joblib.load(os.path.join(model_directory_path, "model.joblib"))

    yield  # Signal readiness to the runner.

    for x_historical, x_online in datasets:
        # Summarize the historical segment once.
        x_h = np.asarray(x_historical, dtype=np.float64)
        mu_h = float(x_h.mean()) if len(x_h) else 0.0
        sd_h = float(x_h.std(ddof=1)) if len(x_h) > 1 else 1.0
        sd_h = max(sd_h, 1e-8)  # avoid divide-by-zero

        # Streaming state, one scalar each.
        mu_ewma = mu_h    # Start centered on the historical mean.
        n_eff   = 0.0     # Effective sample size of the EWMA.

        for point in x_online:
            x = float(point)

            # O(1) EWMA update.
            mu_ewma = (1.0 - ALPHA) * mu_ewma + ALPHA * x
            n_eff   = (1.0 - ALPHA) * n_eff   + 1.0  # Grows to 1/alpha.

            # z-score of the running EWMA mean vs the historical mean.
            se = sd_h / math.sqrt(max(n_eff, 1.0))
            z  = (mu_ewma - mu_h) / max(se, 1e-8)

            # Squash to [0, 1] without saturating.
            score = math.tanh(abs(z) / KAPPA)

            yield float(score)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def score_series(x_hist, x_online, model_dir="resources"):
    """Feeds one series through infer() and returns the score array."""
    gen = infer([(x_hist, iter(x_online))], model_dir)
    next(gen)  # Handshake yield
    return np.array([next(gen) for _ in x_online])

def plot_break_test(x_online, scores, tau, title="Break Test"):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
        
    # 1. Observations
    ax1.plot(x_online, label="Online observations", color="steelblue", lw=1.2)
    ax1.axvline(tau, color="red", linestyle="--", label="True break(tau)")
    ax1.set_ylabel("Value")
    ax1.legend(loc="upper left")
    ax1.set_title(title)
    
    # 2. Predicted Break Probability Scores
    ax2.plot(scores, label="Detector score", color="darkorange", lw=1.8)
    ax2.axvline(tau, color="red", linestyle="--")
    ax2.set_ylabel("Score [0, 1]")
    ax2.set_xlabel("Online Step")
    ax2.set_ylim(-0.05, 1.05)
    ax2.legend(loc="upper left")
    
    plt.tight_layout()
    plt.show()

In [ ]:
rng = np.random.default_rng(42)
N_HIST = 1000
N_PRE  = 50   # Online steps before break
N_POST = 100  # Online steps after break
TAU = N_PRE   # Break happens at index 50

x_hist = rng.normal(0, 3, size=N_HIST)
pre_break = rng.normal(0, 3, size=N_PRE)

post_break = rng.normal(loc=1.5, scale=1.0, size=N_POST)
x_online = np.concatenate([pre_break, post_break])
scores_mean = score_series(x_hist, x_online)
plot_break_test(x_online, scores_mean, TAU, "Mean Shift Test")


# Variance shifts from 1.0 to 3.0 (mean remains 0.0)
post_break = rng.normal(loc=0.0, scale=1.0, size=N_POST)
x_online = np.concatenate([pre_break, post_break])
scores_var = score_series(x_hist, x_online)

plot_break_test(x_online, scores_var, TAU, "Variance Shift Test")

In [ ]:
crunch_tools.test(
    # Uncomment to skip re-training each time
    # force_first_train=False,

    # Uncomment to skip the determinism check
    # no_determinism_check=True,
)

In [ ]:
prediction = pd.read_parquet("prediction/prediction.parquet")

# Load the ground-truth labels supplied with the local tester.
y_test = pd.read_parquet("data/y_test.reduced.parquet")

# Merge predictions with true labels on (id, time).
merged = prediction.merge(
    y_test,
    how="left",
    left_index=True,
    right_index=True,
)

# Add the online step index (0, 1, 2, ...).
merged["time_online"] = merged.groupby("id").cumcount()

# Weighted per-step AUC.
weighted_auc_sum = 0.0
total_weight     = 0.0

for t, group in merged.groupby("time_online"):
    labels = group["target"].values
    scores = group["prediction"].values

    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())
    if n_pos == 0 or n_neg == 0:
        continue

    auc_t  = float(roc_auc_score(labels, scores))
    weight = float(n_pos * n_neg)

    weighted_auc_sum += weight * auc_t
    total_weight     += weight

ts_auc = weighted_auc_sum / total_weight if total_weight > 0 else 0.5
print(f"Local TS-AUC: {ts_auc:.4f}")